# Lecture: Same MLP in PyTorch (Autograd + Adam + DataLoader)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain Tensor, Module/forward, and the autograd graph in one sentence each, and trace one step: `zero_grad → forward → loss → backward → step`.
- Explain why gradients accumulate (the 4, 8, 12 bars in §5) and why forgetting `zero_grad()` silently breaks training.
- Contrast the SGD rule with the Adam rule (§6: SGD final loss 0.0138 vs Adam 0.0069) and read DataLoader batch math (§7).

## The big idea

Project 1 made you do calculus by hand. PyTorch hires an accountant to do it for you. Data arrives as a **Tensor** — PyTorch's array, like a NumPy array but able to remember operations for calculus and live on a GPU. A **Module** (`nn.Module`) is a building block that owns weights; its **forward** method runs input through the layers and produces raw scores, while secretly taping every operation into a little receipt roll called the **autograd graph**. Calling **`loss.backward()`** replays those receipts in reverse with the chain rule and files each result into a drawer attached to every weight, called **`.grad`**. Quirk: the accountant *adds* each new receipt on top of whatever is already in the drawer — gradients **accumulate** — so every step starts by emptying the drawers (**`zero_grad()`**). Then the **optimizer** nudges each weight: plain **SGD** uses one fixed step size for all weights, while **Adam** keeps a running average of recent gradients (direction, called momentum) and of squared gradients (noisiness) and gives each weight its own adaptive step — which is why Adam's curve ends steadier and lower here. A **DataLoader** slices the dataset into shuffled **mini-batches** so each step sees fresh data, and **`CrossEntropyLoss`** takes raw scores (logits) — never pre-softmaxed — because it applies log-softmax internally.

In [ ]:
import numpy as np  # bring in numpy to stand in for torch tensors (lecture uses numpy only)
import matplotlib.pyplot as plt  # bring in pyplot for the shape-flow diagram
np.random.seed(0)  # fix the seed so the demo is identical every run
B, N_IN, H, CL = 8, 784, 128, 10  # define dims: batch 8, 784 pixels in, 128 hidden, 10 classes
X = np.random.randn(B, N_IN).astype(np.float32)  # fake one mini-batch of digit images (8x784)
W1 = np.random.randn(N_IN, H).astype(np.float32) * 0.01  # fake first weights: small random values
b1 = np.zeros(H, dtype=np.float32)  # fake first bias: start at zero for clarity
H1 = np.maximum(0, X @ W1 + b1)  # forward steps 1+2: Linear multiply-add then ReLU bend
W2 = np.random.randn(H, CL).astype(np.float32) * 0.01  # fake second weights: hidden 128 -> 10 scores
b2 = np.zeros(CL, dtype=np.float32)  # fake second bias: start at zero for clarity
logits = H1 @ W2 + b2  # forward step 3: hidden activations to raw class scores (8x10)
print('X', X.shape, '-> H1', H1.shape, '-> logits', logits.shape)  # print the pipeline: 8x784 -> 8x128 -> 8x10
print('logits row 0:', np.round(logits[0], 3))  # print one raw-score row (ranking only, §7 turns it into probs)
print('params:', W1.size + b1.size + W2.size + b2.size)  # print trainable count: 784*128+128+128*10+10 = 101770
fig, ax = plt.subplots(figsize=(8, 2.4))  # make a wide flat figure for the shape-flow diagram
ax.axis('off')  # hide axes: this is a diagram, not a plot
ax.set_xlim(0, 12)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
boxes = [('batch X\n8x784', 1.2), ('Linear 784->128\n@W1+b1', 4.0), ('ReLU\n(bend)', 6.4), ('Linear 128->10\nlogits', 9.0), ('CE loss\n§7', 11.0)]  # stage name + x-center
for name, xc in boxes:  # draw each stage box in order
    ax.text(xc, 1.5, name, ha='center', va='center', fontsize=8,  # centered stage label text
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box around the label
for xa, xb in [(2.2, 3.0), (5.0, 5.7), (7.1, 7.9), (10.1, 10.4)]:  # arrow gaps between consecutive boxes
    ax.annotate('', xy=(xb, 1.5), xytext=(xa, 1.5),  # draw a forward arrow from left box to right box
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.set_title('Anchor: one mini-batch flows X -> hidden -> logits -> loss (shapes above)')  # title the pipeline story
plt.tight_layout()  # snug the layout so labels are not clipped
plt.show()  # render the diagram

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 builds the prediction, §5 tapes and replays the calculus, §6 nudges the weights, §7 feeds the data.

| Term | One-liner | Section |
|---|---|---|
| Tensor | PyTorch's array: like NumPy, but remembers ops for calculus and can live on GPU | §4 |
| Module (`nn.Module`) | A model block that owns weights and defines `forward` | §4 |
| forward | Running input through the model to get logits (tapes the graph behind the scenes) | §4 |
| autograd graph | The taped receipts of the forward pass; `backward()` replays them in reverse | §5 |
| `backward()` | The call that fills every weight's `.grad` via the chain rule | §5 |
| `.grad` / `zero_grad()` | `.grad` stores a gradient and accumulates; `zero_grad()` empties it each step | §5 |
| `step()` (SGD) | The update itself: `w -= lr * grad`, one fixed step size | §6 |
| Adam | Adaptive stepper: per-weight momentum + noisiness averages (ends 0.0069 vs SGD 0.0138) | §6 |
| DataLoader / mini-batch | The helper slicing the dataset into shuffled batches (lab: 64 at a time) | §7 |
| CrossEntropyLoss (logits in!) | The loss; applies log-softmax internally, so feed it RAW logits | §7 |

### §4 Tensor, Module, and forward

**In one sentence: a Tensor is PyTorch's calculus-aware array, a Module is a block that owns weights and defines `forward`, and `forward` turns one mini-batch into raw class scores (logits) while taping the autograd graph behind the scenes.**

A Tensor looks like a NumPy array (`shape`, `dtype`) with two superpowers: it can live on the GPU and it can remember what made it (the tape §5 replays). `nn.Linear(n_in, n_hid)` owns one weight matrix plus one bias vector — exactly Project 1's `W, b` — and `nn.ReLU()` owns nothing (pure bend). The lab's model is `fc1 → relu → fc2`, the same 784 → 128 → 10 shape as Project 1 (101770 trainable numbers) so the comparison is fair. `forward(xb)` returns raw logits with shape `(batch, 10)`: no softmax here, because `CrossEntropyLoss` does that internally (§7). Watch it: the diagram pins shapes to stations, then tiny numbers walk one batch through both Linear steps.

In [ ]:
import numpy as np  # reuse numpy for the tiny forward math
import matplotlib.pyplot as plt  # reuse pyplot for the tensor-op diagram
np.random.seed(2)  # fix the seed for reproducibility
X = np.array([[0.5, -1.0]])  # define one sample with 2 features (shape 1x2, toy stand-in for 1x784)
W = np.array([[1.0, 0.5, -2.0], [0.5, 1.5, 1.0]])  # define toy weights: 2 inputs -> 3 hidden (shape 2x3)
b = np.array([[0.1, -0.2, 0.0]])  # define one bias per hidden unit (shape 1x3)
h = np.maximum(0, X @ W + b)  # forward mini-demo: Linear multiply-add, then ReLU bend (negatives to zero)
print('X shape', X.shape, '-> h shape', h.shape)  # print the shape change: columns 2 -> 3
print('h:', np.round(h, 3))  # print hidden values: [0.1, -1.7, -2.0] bent to [0.1, 0, 0]
print('tensor = array + device + tape; Module = weights + forward recipe')  # print the one-line distinction
fig, ax = plt.subplots(figsize=(8, 2.4))  # make a wide flat figure for the tensor-op diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
stages = [('Tensor xb\n(batch, 784)', 1.5), ('fc1: Linear\n@W1+b1', 4.2), ('relu\nmax(0, .)', 6.6), ('fc2: Linear\n@W2+b2', 9.0), ('logits\n(batch, 10)', 11.0)]  # stage names in flow order
for name, xc in stages:  # draw each stage box left to right
    ax.text(xc, 1.5, name, ha='center', va='center', fontsize=8,  # centered stage label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box for each station
for xa, xb in [(2.6, 3.4), (5.0, 5.8), (7.3, 8.2), (9.9, 10.4)]:  # gaps between consecutive stage boxes
    ax.annotate('', xy=(xb, 1.5), xytext=(xa, 1.5),  # draw the forward arrow left to right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.set_title('Tensor-op map: Module owns W,b; forward tapes every op for §5')  # title the ownership story
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §5 Autograd graph, backward, grad, and zero_grad

**In one sentence: `forward` tapes every operation into the autograd graph, `backward()` replays that tape in reverse with the chain rule into each weight's `.grad` drawer, and because new gradients ADD onto old ones you must call `zero_grad()` first every step.**

Tiny example used below: input `x = 2.0`, weight `w = 0.5`, hidden `h = w*x = 1.0`, output `y = h² = 1.0` (the squaring stands in for the loss). Backward from the seed `dL/dy = 1`: `dL/dh = 2h = 2.0`, then `dL/dw = dL/dh × x = 4.0`. That 4.0 is the one true gradient — but watch the drawers: forget `zero_grad()` and three steps file 4, then 4+4 = 8, then 12 (stale sums steer the optimizer); call it and every step files a fresh 4, 4, 4. That is the whole bug the lab makes you feel on purpose. Watch it: the numbers first, then the 4/8/12 bars vs the flat 4/4/4 bars.

In [ ]:
import numpy as np  # reuse numpy for the gradient-buffer simulation
import matplotlib.pyplot as plt  # reuse pyplot for the accumulation bar chart
np.random.seed(2)  # fix the seed (values below are fixed; seed keeps the habit)
x = 2.0  # define a toy input scalar for the tiny computation graph
w = 0.5  # define a toy weight scalar we take the gradient of
h = w * x  # forward step 1: hidden value h = w*x = 1.0 (taped as a receipt)
y = h ** 2  # forward step 2: output y = h^2 = 1.0 (the squaring stands in for the loss)
dy = 1.0  # seed gradient dL/dy = 1: every backward pass starts from 1
dh = dy * 2.0 * h  # chain-rule step 1: dy/dh = 2h, so dL/dh = 1*2*1.0 = 2.0
dw = dh * x  # chain-rule step 2: dh/dw = x, so dL/dw = 2.0*2.0 = 4.0
print('forward: h =', h, 'y =', y)  # print the forward values flowing left to right
print('backward: dL/dh =', dh, 'dL/dw =', dw)  # print the gradients flowing right to left (true grad 4.0)
true_grad = 4.0  # store the correct gradient computed above for the buffer demo
grads_nozero = []  # list for the buffer when we FORGET zero_grad (it accumulates)
buf = 0.0  # the .grad drawer starts empty like a fresh parameter
for step in range(3):  # simulate 3 training steps reusing the same drawer
    buf = buf + true_grad  # ACCUMULATE: PyTorch adds each new gradient onto .grad by default
    grads_nozero.append(buf)  # record the drawer content used for this step's update
grads_zeroed = []  # list for the buffer when we CALL zero_grad each step
for step in range(3):  # simulate the same 3 steps with proper clearing
    buf = 0.0  # ZERO_GRAD: reset the drawer before backward (standard loop hygiene)
    buf = buf + true_grad  # add only this step's fresh gradient onto the clean drawer
    grads_zeroed.append(buf)  # record the drawer content used for this step's update
print('without zero_grad buffers:', grads_nozero)  # print [4.0, 8.0, 12.0]: stale gradients pile up
print('with zero_grad buffers:', grads_zeroed)  # print [4.0, 4.0, 4.0]: each step uses fresh gradient
print('graph receipts: h=w*x, y=h^2; backward replays y->h->w in reverse')  # print the tape moral
labels = ['step 1', 'step 2', 'step 3']  # x labels for the three simulated steps
xs = np.arange(3)  # numeric x positions 0,1,2 for grouped bars
width = 0.35  # bar width so the two groups sit side by side
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.bar(xs - width / 2, grads_nozero, width, label='forgot zero_grad (4, 8, 12)')  # draw the growing stale-gradient bars
ax.bar(xs + width / 2, grads_zeroed, width, label='zero_grad each step (4, 4, 4)')  # draw the flat fresh-gradient bars
ax.set_xticks(xs)  # place tick marks at the three step positions
ax.set_xticklabels(labels)  # name each tick after its training step
ax.set_ylabel('.grad drawer used by optimizer')  # label y as the gradient value fed to the update
ax.set_title('zero_grad clears stale gradients (they accumulate by default)')  # plain-English title
ax.legend()  # show which color is which habit
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure inline

### §6 Step and Adam — the two nudgers

**In one sentence: `step()` nudges each weight using its `.grad`, plain SGD with one fixed rule `w -= lr × grad` for every weight, Adam with a per-weight adaptive rule `w -= lr × m̂ / (√v̂ + eps)` where `m̂` averages recent gradients (direction) and `v̂` averages squared gradients (noisiness).**

Same noisy valley race for both (40 steps, learning rate 0.2, identical mini-batch noise): SGD bounces because one step size must serve calm and jumpy directions alike, while Adam damps jumpy weights and strides steady ones. Final true losses: SGD 0.0138 vs Adam 0.0069, and the last-5-step average (the wiggle level near the minimum) is 0.0697 for SGD but only 0.004 for Adam — Adam ends lower *and* steadier. That is why the lab trains 10 epochs with Adam instead of 15 with SGD.

> Teaching simplification: the toy below inits Adam's second moment at `v=1.0` instead of real Adam's `v=0` + bias correction, so early steps are not shrunk ~1000× — the SGD-vs-Adam *ordering* transfers, the exact 0.0069 does not. Real `torch.optim.Adam` starts at zero. Watch it: the two loss curves on a log scale so the early plunge and the late wiggle are both visible.

In [ ]:
import numpy as np  # bring in numpy for the optimization simulation
import matplotlib.pyplot as plt  # bring in pyplot for the SGD-vs-Adam plot
np.random.seed(7)  # fix the seed so the noisy gradients are identical every run
true_min = 3.0  # define the valley bottom both optimizers chase (minimum at w=3)
steps = 40  # define the number of update steps each optimizer gets
noise = np.random.randn(steps) * 0.8  # draw fixed gradient noise mimicking mini-batch sampling noise
w_sgd = 0.0  # define SGD start weight far from the minimum
w_adam = 0.0  # define Adam start weight at the same spot for a fair race
lr_sgd = 0.2  # define SGD learning rate: one fixed step size for every parameter
lr_adam = 0.2  # define Adam base step size (adapted per step by the running averages)
b1, b2 = 0.9, 0.999  # define Adam decay rates: memory length for the two running averages
m = 0.0  # define Adam running average of gradients (momentum term), starts at zero
v = 1.0  # define Adam running average of squared gradients (scale term), starts at one
hist_sgd = [w_sgd]  # create list recording the SGD path, starting at w=0
hist_adam = [w_adam]  # create list recording the Adam path, starting at w=0
for t in range(1, steps + 1):  # loop over update steps, counting from 1 for Adam's correction
    g = 2.0 * (w_sgd - true_min) + noise[t - 1]  # compute noisy SGD gradient: true slope plus sampling noise
    w_sgd = w_sgd - lr_sgd * g  # take one SGD step with the fixed learning rate
    hist_sgd.append(w_sgd)  # record the new SGD weight for the plot
    ga = 2.0 * (w_adam - true_min) + noise[t - 1]  # compute the same noisy gradient for Adam (same noise = fair)
    m = b1 * m + (1 - b1) * ga  # update Adam momentum: blend old average with the new gradient
    v = b2 * v + (1 - b2) * (ga ** 2)  # update Adam scale: blend old average with the squared gradient
    mhat = m / (1 - b1 ** t)  # correct momentum for startup bias (early averages start near zero)
    vhat = v / (1 - b2 ** t)  # correct the scale for startup bias the same way
    w_adam = w_adam - lr_adam * mhat / (np.sqrt(vhat) + 1e-8)  # take one Adam step: direction from mhat, size from vhat
    hist_adam.append(w_adam)  # record the new Adam weight for the plot
hist_sgd = np.array(hist_sgd)  # convert the SGD path to an array for loss math
hist_adam = np.array(hist_adam)  # convert the Adam path to an array for loss math
loss_sgd = (hist_sgd - true_min) ** 2  # compute noise-free loss along the SGD path
loss_adam = (hist_adam - true_min) ** 2  # compute noise-free loss along the Adam path
print('SGD final loss:', round(float(loss_sgd[-1]), 4))  # print SGD final true loss: 0.0138
print('Adam final loss:', round(float(loss_adam[-1]), 4))  # print Adam final true loss: 0.0069 (lower)
print('SGD last-5 mean loss:', round(float(loss_sgd[-5:].mean()), 4))  # print SGD wiggle level: 0.0697
print('Adam last-5 mean loss:', round(float(loss_adam[-5:].mean()), 4))  # print Adam wiggle level: 0.004 (steadier)
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.plot(loss_sgd, 'o-', markersize=3, label='SGD fixed lr=0.2')  # draw the bouncier SGD loss curve
ax.plot(loss_adam, 's-', markersize=3, label='Adam adaptive lr')  # draw the smoother Adam loss curve
ax.set_xlabel('update step')  # label x as the optimizer step number
ax.set_ylabel('true loss (w-3)^2')  # label y as the noise-free loss value
ax.set_title('Adam adapts each step; SGD uses one fixed step size')  # plain-English title
ax.set_yscale('log')  # use a log y-scale so fast drops and small wiggles are both visible
ax.legend()  # show the legend mapping colors to optimizers
ax.grid(True, which='both', alpha=0.3)  # add a faint grid on both log ticks
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure inline

### §7 DataLoader and CrossEntropyLoss (logits in!)

**In one sentence: a DataLoader slices the dataset into shuffled mini-batches (the lab: batch 64, so 8 toy samples at batch 4 give 2 updates per epoch), and CrossEntropyLoss scores raw logits directly because it applies log-softmax internally — feeding it pre-softmaxed probabilities double-normalizes and silently breaks training.**

Batch math to memorize: updates per epoch = dataset ÷ batch (8 ÷ 4 = 2 below; the lab's 60000 ÷ 64 ≈ 938, times 10 epochs ≈ 9380 Adam updates). `shuffle=True` re-deals the slices every epoch so each step sees a fresh mix. Loss demo on one sample with logits `[2.0, 1.0, -1.5]` and true class 0: softmax gives `[0.715, 0.263, 0.022]`, and the loss is `-log(0.715) = 0.3351` — only the true class's probability matters. The lab's `evaluate()` wraps test loops in `no_grad()` so evaluation tapes no graph. Watch it: the batch-slice diagram plus the logits → loss numbers.

In [ ]:
import numpy as np  # reuse numpy for the softmax + loss math
import matplotlib.pyplot as plt  # reuse pyplot for the batch-slice diagram
np.random.seed(4)  # fix the seed for reproducibility
N, B, E = 8, 4, 2  # toy setup: 8 samples, batch 4, 2 epochs (lab uses 60000/64/10)
updates = E * (N // B)  # total weight updates = epochs times batches-per-epoch = 4
print('toy: 8 samples / batch 4 x 2 epochs =', updates, 'updates')  # print the counting rule on the toy
print('lab: 10 x (60000 / 64) ~= 9380 Adam updates')  # print the lab-scale version of the same math
logits = np.array([[2.0, 1.0, -1.5]])  # define raw scores for 3 classes (unconstrained numbers, §4 output)
shifted = logits - logits.max(axis=1, keepdims=True)  # subtract row max: overflow safety, cancels in division
probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)  # softmax to probabilities (what CE does inside)
loss = -np.log(probs[0, 0])  # cross-entropy for true class 0: -log(probability of the answer)
print('probs:', np.round(probs[0], 3))  # print [0.715 0.263 0.022]: winner amplified vs raw 2:1 gap
print('CE loss (true class 0):', round(float(loss), 4))  # print 0.3351: -log(0.715)
print('rule: feed RAW logits to CE; never softmax first (it double-normalizes)')  # print the golden rule
fig, ax = plt.subplots(figsize=(7, 3.5))  # make a figure for the batch-slice diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 10)  # fix placement range
for e in range(E):  # draw one row per epoch
    ax.text(0.2, 8 - e * 3.4, f'epoch {e + 1}:', va='center', fontsize=9)  # label the epoch row
    for i in range(N):  # draw one square per sample
        ax.add_patch(plt.Rectangle((2 + i * 0.9, 7.3 - e * 3.4), 0.7, 1.2,  # position this sample box
                                      facecolor='lightgreen' if i < B else 'wheat'))  # color batch 1 green, batch 2 wheat
ax.text(5, 0.6, 'green = batch 1, wheat = batch 2 (2 updates per epoch, reshuffled)', ha='center', fontsize=9)  # legend line
ax.set_title('DataLoader: shuffled slices of 4; CE scores raw logits per slice')  # title the feeding story
plt.tight_layout()  # snug the layout
plt.show()  # render the figure

## How it all fits together — the system map §8

One training step is a loop with every § term playing a position. **Make tensors** (§4): a DataLoader slice lands as `xb, yb`. **Forward** (§4): `model(xb)` runs `fc1 → relu → fc2` and tapes the graph. **Score** (§7): `CrossEntropyLoss` eats raw logits. **Backward** (§5): `loss.backward()` replays the tape into every `.grad` — this is Project 1's hand calculus, done by the accountant. **Step** (§6): Adam or SGD reads the drawers and nudges each weight. **Clear** (§5): `zero_grad()` empties the drawers so the next step starts fresh. The map below pins each term to its station; the numbers walk one scalar weight through the analytic gradient and a finite-difference check that plays the role of autograd.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.8))  # wide figure for the left-to-right loop pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('batch\nxb, yb\n§7', 1.2), ('forward\nmodel(xb)\n§4', 3.6), ('loss\nCE(logits)\n§7', 6.0), ('backward\n.fill .grad\n§5', 8.4), ('step\nSGD/Adam\n§6', 10.8), ('zero_grad\nempty drawers\n§5', 13.0)]  # loop stations in order
for name, xc in stages:  # draw each loop station box
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = forward-flow stations
for xa, xb in [(2.0, 2.8), (4.4, 5.2), (6.8, 7.6), (9.2, 10.0), (11.6, 12.3)]:  # gaps between stations
    ax.annotate('', xy=(xb, 2.5), xytext=(xa, 2.5), arrowprops=dict(arrowstyle='->'))  # forward arrows left to right
ax.annotate('', xy=(0.6, 1.2), xytext=(13.0, 1.2), arrowprops=dict(arrowstyle='->', color='red', lw=2))  # long red loop-back arrow
ax.text(7, 3.6, 'ONE STEP: forward tapes, backward replays, step nudges', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.5, 'autograd does P1 hand calculus; zero_grad keeps steps fresh', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = the two hygiene lessons
ax.set_title('System map §8: the five-call loop around every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the end-to-end numeric trace
np.random.seed(9)  # fix the seed (numbers below are hardcoded; habit kept)
w, x = 0.5, 2.0  # one weight and one input (the §5 toy, now traced as a full step)
h = w * x  # FORWARD: hidden = 0.5*2.0 = 1.0 (taped receipt 1)
loss = h ** 2  # SCORE: squared stand-in loss = 1.0 (stands in for CE on logits)
analytic = 2.0 * h * x  # BACKWARD by hand: dL/dw = 2h*x = 4.0 (chain rule, §5)
e = 1e-4  # finite-difference nudge for the autograd stand-in check
numeric = ((w + e) * x) ** 2 - ((w - e) * x) ** 2  # difference of losses at w+e and w-e
numeric = numeric / (2 * e)  # divide by 2e: the centered-difference gradient estimate
print('loss:', loss)  # print 1.0: the single wrongness number
print('manual chain-rule dL/dw:', analytic)  # print 4.0: what P1 computed by hand
print('autograd-stand-in dL/dw:', round(float(numeric), 4))  # print ~4.0: what backward() files into .grad
lr = 0.1  # step size for the demo update
w_new = w - lr * analytic  # STEP: one SGD nudge w -= lr*grad: 0.5 -> 0.1
print('w before/after one SGD step:', w, '->', round(float(w_new), 4))  # print the downhill nudge (then zero_grad!)

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, with PyTorch doing the taping:

1. **Cell 1 (goal markdown)** — states the mission: rebuild Project 1 with `Linear/ReLU/CE/Adam`. You already know the loop (§8).
2. **Cell 2 (setup)** — seeds plus `torch.manual_seed(42)` and the CUDA-availability log. Same seed habit as this lecture.
3. **Cell 3 (`load_tensors`)** — same MNIST-or-digits fallback, returned as Tensors (§4) instead of NumPy arrays.
4. **Cell 4 (`MLP`, loss, optimizer)** — `nn.Linear → ReLU → nn.Linear`, `CrossEntropyLoss` (logits in, §7!), `Adam(lr=1e-3)` (§6). Same 128-wide shape as Project 1 (101770 params) for a fair race.
5. **Cell 5 (manual single step)** — the loop dissected on 8 samples: `zero_grad → forward → loss → backward → step → zero_grad` (§8), with gradient norms printed. This is the 4/8/12 bars (§5) in real code.
6. **Cell 6 (training loop)** — the same five calls inside a `DataLoader` mini-batch loop (§7) for 10 epochs ≈ 9380 updates, with a `no_grad()` evaluation helper (evaluation must not tape graphs).
7. **Cell 7 (save + plot)** — `results/metrics_pytorch.csv` and loss/accuracy curves; compare epoch counts against Project 1's SGD (§6 explains the gap: Adam 0.0069 vs SGD 0.0138).
8. **Cell 8 (README prompts)** — explain grad accumulation (§5), the backward graph (§5), and why Adam beat SGD (§6); bonus: rerun with SGD(lr=0.1) and tabulate.

## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Where does `.grad` come from, and why must you clear it?**

*Answer: from `loss.backward()` replaying the autograd graph in reverse (§5) — the chain rule on the taped receipts, the same 4.0 this lecture traces in §8. New gradients ADD onto `.grad`, so without `zero_grad()` steps use stale sums (4, 8, 12…) instead of fresh 4s.*

2. **What do `forward`, `loss.backward()`, and `opt.step()` each do, in one sentence?**

*Answer: `forward` runs the batch through the Module and tapes the graph (§4); `backward()` replays the tape into every `.grad` (§5); `step` reads the drawers and nudges each weight — SGD with one fixed size, Adam adaptively (§6).*

3. **Why does Adam beat plain SGD here, and what must you never feed `CrossEntropyLoss`?**

*Answer: Adam adapts each weight's step from running averages of gradients and squared gradients, ending lower and steadier (final 0.0069 vs 0.0138, wiggle 0.004 vs 0.0697 in §6). Never feed CE pre-softmaxed probabilities — it wants RAW logits (§7), or it double-normalizes.*